1\. Environment Setup
---------------------

The authors utilized a Linux-based server with NVIDIA RTX GPUs. For this replication, we use **Detectron2**, a standard library for object detection research.

**Kaggle Note:** We install Detectron2 from source to ensure compatibility with Kaggle's pre-installed PyTorch version. We also ensure pyyaml is pinned to prevent dependency conflicts.

In [1]:
# Installation (Uncomment if needed)
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu111/torch1.9/index.html

!pip install effdet timm -q

import os
import copy
import torch
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
from datetime import datetime
from PIL import Image



# Define Output Directory for Kaggle (Writable path)
OUTPUT_DIR = "/kaggle/working/output/naive_ddl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Using Torch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 3.5 MB/s eta 0:00:00
Using Torch version: 2.8.0+cu126 | CUDA available: True


2\. Dataset Registration
------------------------

As per **Section 3.1**, the dataset consists of 5,900 images of paprika plants. We follow the paper's split:

*   **70% Training**
    
*   **10% Validation**
    
*   **20% Testing**
    

The DDL unit focuses on 6 abnormality categories found in the Paprika dataset.

In [ ]:
import os
import cv2
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class PaprikaEfficientDetDataset(Dataset):
    def __init__(self, annotation_file, image_dir, image_size=(512, 512)):
        """
        Args:
            annotation_file: Path to your COCO-style JSON annotation file.
            image_dir: Path to the folder containing the images.
            image_size: The target size to resize images for EfficientDet.
        """
        self.image_dir = image_dir
        self.image_size = image_size
        
        # Load the annotations
        with open(annotation_file, 'r') as f:
            self.coco_data = json.load(f)
            
        self.images = {img['id']: img for img in self.coco_data['images']}
        self.image_ids = list(self.images.keys())
        
        # Group annotations by image_id
        self.annotations = {img_id: [] for img_id in self.image_ids}
        for ann in self.coco_data['annotations']:
            self.annotations[ann['image_id']].append(ann)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        img_id = self.image_ids[index]
        img_info = self.images[img_id]
        
        # 1. Load and format the image
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Calculate resize scales
        orig_h, orig_w = image.shape[:2]
        scale_x = self.image_size[0] / orig_w
        scale_y = self.image_size[1] / orig_h
        
        # Resize image to target size (512x512)
        image = cv2.resize(image, self.image_size)
        image = image.astype(np.float32) / 255.0 # Normalize to [0, 1]
        
        # 2. Extract and scale bounding boxes
        bboxes = []
        classes = []
        
        for anno in self.annotations[img_id]:
            # COCO format is [x_min, y_min, width, height]
            x_min, y_min, w, h = anno['bbox']
            
            # Convert to [x_min, y_min, x_max, y_max] and scale to new image size
            x_min = x_min * scale_x
            y_min = y_min * scale_y
            x_max = (x_min + w) * scale_x
            y_max = (y_min + h) * scale_y
            
            # EffDet expects boxes in [y_min, x_min, y_max, x_max] format internally 
            # during training if using standard setups, but the base API accepts standard XYXY
            bboxes.append([x_min, y_min, x_max, y_max])
            
            # Classes in effdet are typically 1-indexed (0 is background)
            # Make sure your category_ids align with this!
            classes.append(anno['category_id'])
            
        # Handle images with no anomalies
        if len(bboxes) == 0:
            bboxes = np.zeros((0, 4), dtype=np.float32)
            classes = np.zeros((0,), dtype=np.int64)
            
        # 3. Format the target dictionary exactly as effdet requires
        target = {
            'bboxes': torch.tensor(bboxes, dtype=torch.float32),
            'cls': torch.tensor(classes, dtype=torch.int64)
        }
        
        # Convert image to PyTorch tensor format: (Channels, Height, Width)
        image = torch.tensor(image).permute(2, 0, 1)

        return image, target

# ==========================================
# Initialize the Dataset
# ==========================================
# IMPORTANT: Update these paths to point to your actual Kaggle input directories
TRAIN_ANN_FILE = "/kaggle/input/your-dataset-name/train_annotations.json"
TRAIN_IMG_DIR = "/kaggle/input/your-dataset-name/train_images/"

train_dataset = PaprikaEfficientDetDataset(TRAIN_ANN_FILE, TRAIN_IMG_DIR)

# A simple custom collate function is needed because images have varying numbers of bounding boxes
def collate_fn(batch):
    images, targets = tuple(zip(*batch))
    images = torch.stack(images)
    return images, targets

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=4, # Adjust based on GPU memory (T4 in Kaggle usually handles 4-8)
    shuffle=True, 
    num_workers=2,
    collate_fn=collate_fn
)

print(f"Dataset loaded! Total training images: {len(train_dataset)}")

In [ ]:
# 6 Classes from the dataset README
# Note: Ensure this order matches the ID (0-5) in the YOLO .txt files.
# Usually YOLO starts at 0. Check your _classes.txt if avail.
CLASS_NAMES = [
    "blossom_end_rot",
    "graymold",
    "powdery_mildew",
    "spider_mite",
    "spotting_disease",
    "snails_and_slugs"
]

# === ADJUST THIS PATH ===
# Based on your screenshot, it is likely:
# /kaggle/input/paprika-dataset/data
# OR /kaggle/input/{your-dataset-name}/data
DATASET_ROOT = "/kaggle/input/datasets/tijesu26/paprika-dataset/data" 

def get_paprika_dicts(img_dir, label_dir):
    """
    Parses YOLO format dataset for Detectron2.
    """
    dataset_dicts = []
    # Find all images (support jpg and png)
    image_files = glob.glob(os.path.join(img_dir, "*.jpg")) + glob.glob(os.path.join(img_dir, "*.png"))
    
    print(f"Found {len(image_files)} images in {img_dir}")
    
    for idx, img_path in enumerate(image_files):
        record = {}
        
        # 1. Get Image Dimensions (Needed for YOLO relative -> absolute conversion)
        # We use PIL to lazy load just the size (faster than reading full image)
        with Image.open(img_path) as img:
            width, height = img.size
            
        record["file_name"] = img_path
        record["image_id"] = idx
        record["height"] = height
        record["width"] = width
        
        # 2. Find Corresponding Label File
        # YOLO structure: images/file.jpg -> labels/file.txt
        filename = os.path.basename(img_path)
        label_filename = os.path.splitext(filename)[0] + ".txt"
        label_path = os.path.join(label_dir, label_filename)
        
        objs = []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                
                # YOLO: class_id, center_x, center_y, width, height (Normalized)
                class_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])
                
                # Conversion to Absolute XYXY
                abs_cx = cx * width
                abs_cy = cy * height
                abs_w = w * width
                abs_h = h * height
                
                x_min = abs_cx - (abs_w / 2)
                y_min = abs_cy - (abs_h / 2)
                x_max = abs_cx + (abs_w / 2)
                y_max = abs_cy + (abs_h / 2)
                
                obj = {
                    "bbox": [x_min, y_min, x_max, y_max],
                    "bbox_mode": BoxMode.XYXY_ABS,
                    "category_id": class_id,
                }
                objs.append(obj)
        
        record["annotations"] = objs
        dataset_dicts.append(record)
        
    return dataset_dicts



In [ ]:
# Verification
if "paprika_train" in DatasetCatalog.list():
    dataset_dicts = DatasetCatalog.get("paprika_train")
    if len(dataset_dicts) > 0:
        d = dataset_dicts[20]
        img = utils.read_image(d["file_name"], format="BGR")
        visualizer = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("paprika_train"), scale=0.5)
                
        # === REDUCE TEXT SIZE ===
        # Manually override the calculated font size
        # Try values between 10 (small) and 25 (large)
        # visualizer._default_font_size = 15
        
        out = visualizer.draw_dataset_dict(d)
        plt.figure(figsize=(10, 10))
        plt.imshow(out.get_image()[:, :, ::-1])
        plt.title("Sample Data Verification")
        plt.show()
    else:
        print("Dataset registered but no images found.")

## 3. Augmentation Strategy

We implement the specific augmentations mentioned in **Table 4** of the paper.
The authors noted that *Color Temperature* and *Noise* reduced performance, while *Geometric* transforms improved it.

**Implemented Transforms:**
1.  **Random Flip:** Probability 0.9 (Horizontal).
2.  **Scale/Resize:** Table 4 lists `Scale x=[0.8, 1.2]`. In Detectron2, we implement this via `ResizeShortestEdge` with a range of sizes to simulate multi-scale training around the 1024px baseline.


## 4. Model Configuration (Naïve DDL)
  - **Optimizer:** SGD
  - **Momentum:** 0.9
  - **Base LR:** 0.08 (with Cosine Decay)
  - **Weight Decay:** 0.0005
  - **Batch Size:** 16
  - **Max Iterations:** 100,000

<!-- end list -->

In [ ]:
def setup_cfg():
    cfg = get_cfg()
    
    # 1. Architecture: Faster R-CNN R50 FPN (Matches Table 5 Naive Framework)
    cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
    
    # 2. Dataset Registry (Updated to match the split names we registered)
    cfg.DATASETS.TRAIN = ("paprika_train",)
    cfg.DATASETS.TEST = ("paprika_valid",) # Using 'valid' as test
    cfg.DATALOADER.NUM_WORKERS = 2  
    
    # 3. Solver / Training Config (Table 3)
    # Note: 16 batch size requires high VRAM. On Kaggle P100 (16GB), use 4 or 8.
    # We scale LR accordingly: 0.08 is for batch 16. If batch 4, LR -> 0.02
    cfg.SOLVER.IMS_PER_BATCH = 4        
    cfg.SOLVER.BASE_LR = 0.02           
    
    cfg.SOLVER.MOMENTUM = 0.9
    cfg.SOLVER.WEIGHT_DECAY = 0.0005
    cfg.SOLVER.MAX_ITER = 10000          # Reduced for Kaggle Demo (Paper: 100000)
    cfg.SOLVER.WARMUP_ITERS = 1000      # 0-1K Warmup
    
    # 4. Learning Rate Scheduler
    cfg.SOLVER.LR_SCHEDULER_NAME = "WarmupCosineLR"
    
    # 5. Input Resolution (Table 5: 1024x1024)
    cfg.INPUT.MIN_SIZE_TRAIN = (1024,)
    cfg.INPUT.MAX_SIZE_TRAIN = 1024
    cfg.INPUT.MIN_SIZE_TEST = 1024
    cfg.INPUT.MAX_SIZE_TEST = 1024
    
    # 6. Model Heads
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512 
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 6  # 6 Paprika anomalies
    
    # 7. Output
    cfg.OUTPUT_DIR = OUTPUT_DIR
    
    return cfg

cfg = setup_cfg()

## 5\. Training Loop

We initialize the custom trainer and start the process.

In [ ]:
# Initialize the trainer with our custom mapper
trainer = NaiveDDLTrainer(cfg)

# Load ImageNet pretrained weights (standard for D2)
trainer.resume_or_load(resume=False)

# Start Training
print(f"Starting Training: {datetime.now()}")
print(f"Configured for {cfg.SOLVER.MAX_ITER} iterations.")

In [ ]:
# Uncomment below to run training
trainer.train()